# Frequency-domain tools — Bode, margins, Nyquist, root locus, step response

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/classical_control/frequency_domain_tools.ipynb)

This notebook is the classical control chapter, read through the [minilink](https://github.com/alx87grd/minilink) API. Every tool follows the same recipe:

1. **Linearize** the plant about an operating point $(\bar x, \bar u)$ — the Jacobians $A = \partial f/\partial x$, $B = \partial f/\partial u$, $C = \partial h/\partial x$, $D = \partial h/\partial u$.
2. **Pick one channel**, an output component from an input component, so the model is a SISO state space $(A, b, c, d)$.
3. **Compute with linear algebra** on those four matrices: eigenvalues for poles, $c\,(j\omega I - A)^{-1} b + d$ for the frequency response, $\operatorname{eig}(A - b K c)$ for the root locus, one matrix exponential for the step response.

The API mirrors the recipe. Each verb reads `tool(x_bar, u_bar, ..., of=<output>, wrt=<input>)`, has a data twin and a `plot_` twin, and renders with `backend="matplotlib"` or `backend="plotly"`. This notebook sets that once as `BACKEND` in the import cell. Loops are wired, not derived by hand: `C >> plant` is the loop gain, `C @ plant` the closed loop.

| what you want | data | plot |
| --- | --- | --- |
| poles, zeros, gain | `pzmap` | `plot_pzmap` |
| transfer function | `transfer_function` | — |
| frequency response | `bode`, `frequency_response` | `plot_bode` |
| gain and phase margins | `margins` | drawn on `plot_bode` |
| Nyquist contour | `nyquist` | `plot_nyquist` |
| closed-loop poles vs gain | `root_locus` | `plot_root_locus` |
| unit-step response | `step_response`, `step_info` | `plot_step_response` |
| the loop itself | `C >> plant`, `C @ plant`, `feedback` | `plot_diagram` |

We use the pendulum hanging down as the plant, a PID as the compensator, and the inverted pendulum (with a lead) for the root-locus lesson.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import PID, InvertedPendulum, Lead, Pendulum
from minilink.analysis import step_info

BACKEND = "matplotlib"  # or "plotly" (hover; the one to use in Colab)
# BACKEND = "plotly"

## 1. From a nonlinear plant to a linear channel

The pendulum is $\;(m l^2 + I)\,\ddot\theta = \tau - m g l \sin\theta - d\,\dot\theta$, a nonlinear system with state $x = [\theta, \dot\theta]$, input $u = \tau$ and output $y = x$. Hanging down, $\bar x = [0, 0]$ and $\bar u = 0$ is an equilibrium. Linearizing there gives

$$\dot x = A x + B u, \qquad y = C x + D u,$$

and the channel from the torque to the angle is the transfer function

$$G(s) = c\,(sI - A)^{-1} b + d = \frac{k}{s^2 + 2\zeta\omega_n s + \omega_n^2}.$$

`linearize` returns the four matrices as an `LTISystem`; `pzmap` returns the poles (eigenvalues of $A$), the transmission zeros and the leading gain $k$; `transfer_function` packs them into a `TransferFunction` block.

In [ ]:
plant = Pendulum()
plant.params["d"] = 0.5  # a little damping, so the resonance is finite

x_bar = np.array([0.0, 0.0])  # hanging down, at rest
plant.x0 = x_bar  # the operating point every analysis tool defaults to

lin = plant.linearize(x_bar)
print("A =\n", np.round(lin.A(), 3))
print("B =\n", np.round(lin.B(), 3))

zeros, poles, gain = plant.pzmap(x_bar)  # channel: theta from tau (defaults)
print("poles:", np.round(poles, 3), " zeros:", zeros, " gain:", gain)

G = plant.transfer_function(x_bar)
print(G.name, "  num:", np.round(G.numerator, 3), " den:", np.round(G.denominator, 3))

The two poles at $-0.125 \pm 2.21j$ are the eigenvalues of $A$: a lightly damped oscillator with $\omega_n = \sqrt{4.905} \approx 2.21$ rad/s and $\zeta \approx 0.06$. The pole-zero map draws them in the $s$-plane, `x` for poles and `o` for zeros.

In [ ]:
plant.plot_pzmap(x_bar, backend=BACKEND)

## 2. Frequency response and the Bode diagram

Drive the linear channel with $u = \sin\omega t$: in steady state the output is a sinusoid at the same frequency, scaled by $|G(j\omega)|$ and shifted by $\angle G(j\omega)$. The Bode diagram plots both against $\omega$ on a log axis, the magnitude in decibels, $20\log_{10}|G|$.

For the pendulum the magnitude is flat at low frequency (the static gain $k/\omega_n^2$), peaks at the resonance, then falls at $-40$ dB per decade; the phase goes from $0$ to $-180°$ through $-90°$ at $\omega_n$. `bode` returns the samples; `plot_bode` draws them. The automatic frequency grid runs one decade below the slowest pole or zero to one decade above the fastest.

In [ ]:
w, magnitude_db, phase_deg = plant.bode(x_bar, w=[0.5, 2.21, 10.0])
for wk, mk, pk in zip(w, magnitude_db, phase_deg):
    print(f"w = {wk:5.2f} rad/s   |G| = {mk:7.2f} dB   phase = {pk:8.2f} deg")

plant.plot_bode(x_bar, margins=False, backend=BACKEND)

Set `BACKEND` at the top to `"plotly"` for the interactive figure (hover a point for $\omega$, $|G|$ and the phase). That is the backend to use in Colab.

## 3. The loop gain and the stability margins

The compensator is a **PID** with a filtered derivative, written on the tracking error $e$:

$$C(s) = K_p + \frac{K_i}{s} + K_d\,\frac{s}{\tau s + 1}.$$

The block has one input $e$ and one command $u$. Wiring it in series with the plant, `L = C >> plant`, builds the **loop gain** $L(s) = C(s)\,H(s)$ as an ordinary diagram: input $e$, output the plant's $y$, the feedback path left open. Every analysis verb works on that diagram exactly as on a block, since a diagram linearizes like anything else. The margins read how far $L$ is from the critical point $-1$:

- the **gain margin** is $-20\log_{10}|L(j\omega_{pc})|$ at the phase crossover $\angle L = -180°$: how much the gain can grow before instability;
- the **phase margin** is $180° + \angle L(j\omega_{gc})$ at the gain crossover $|L| = 1$: how much delay the loop tolerates.

In [ ]:
for Kp in (10.0, 20.0, 40.0):
    L = PID(Kp=Kp, Ki=10.0, Kd=2.0, tau=0.05) >> plant  # e -> u -> y: L = C H
    m = L.margins()
    print(f"Kp = {Kp:5.0f}:  phase margin = {m.phase_margin_deg:5.1f} deg "
          f"at {m.w_gain_crossover:4.2f} rad/s,  gain margin = {m.gain_margin_db}")

Raising $K_p$ moves the gain crossover to the right and eats phase margin: the loop is faster and less damped. On this pendulum the phase of $L$ never quite reaches $-180°$, so the gain margin is infinite and the phase margin alone measures the damping. `plot_bode` on the loop-gain diagram draws the crossover frequencies and prints the margins in the corner.

In [ ]:
C = PID(Kp=20.0, Ki=10.0, Kd=2.0, tau=0.05)
L = C >> plant
L.plot_diagram()
L.plot_bode(backend=BACKEND)

The Nyquist diagram is the same response drawn as a curve in the complex plane, $L(j\omega)$ for $\omega$ from $0$ to $\infty$ (solid) and its mirror image for negative frequencies (dashed). The **Nyquist criterion** counts encirclements of $-1$: with a stable $L$, the closed loop is stable when the contour does not encircle the critical point. The distance from the contour to $-1$ is the margin seen geometrically.

In [ ]:
L.plot_nyquist(backend=BACKEND)

## 4. Root locus

The closed-loop poles are the roots of $1 + K\,L(s) = 0$. The root locus draws them in the $s$-plane as the gain $K$ sweeps from $0$ (the open-loop poles, `x`) to $\infty$ (the open-loop zeros, `o`, and the asymptotes). minilink computes each point as $\operatorname{eig}(A - b K c)$ of the channel's state-space model, here the linearization of the whole series diagram, and joins the branches.

On the PID-compensated pendulum the open-loop poles are the plant pair, the integrator at the origin, and the derivative-filter pole at $s = -1/\tau$. The two zeros of $C(s)$ pull branches into the left half-plane: more gain is the same story the phase margin told, seen from the $s$-plane.

In [ ]:
L.plot_root_locus(backend=BACKEND)

gains, roots = L.root_locus()
print("branches:", roots.shape[1], " gain sweep:", f"{gains[1]:.3g} .. {gains[-1]:.3g}")

The inverted pendulum makes the lesson sharper. About the upright, the two open-loop poles are real, $\pm 2.21$. Feeding back the angle alone, $u = -K\theta$, moves them together, they meet at the origin and leave along the imaginary axis: no gain stabilizes the pole. A lead $(s + 2)/(s + 20)$ adds rate feedback; its zero pulls both branches into the left half-plane once $K$ is large enough.

In [ ]:
inverted = InvertedPendulum()
inverted.x0 = np.array([0.0, 0.0])  # upright

inverted.plot_root_locus(backend=BACKEND)  # angle feedback alone

L_lead = Lead(K=1.0, z=2.0, p=20.0) >> inverted
L_lead.plot_root_locus(backend=BACKEND)

gains, roots = L_lead.root_locus()
stable = np.all(roots.real < 0.0, axis=1)
print(f"the lead loop is stable above K = {gains[np.argmax(stable)]:.3g}")

## 5. Closing the loop in minilink and checking the prediction

`T = C @ plant` closes the loop: the operator inserts an `Error` block $e = r - \theta$ (ports `+`, `-`, `e`) in front of the compensator, exposes the reference $r$ as the diagram input, and returns the plant output. The pendulum reports $y = [\theta, \dot\theta]$ while the compensator takes one error, so a `Demux` picks $\theta$; the diagram shows exactly what was wired. The closed loop is itself a system, so the same tools apply: `jacobian("f", "x")` is the closed-loop $A$ matrix, whose eigenvalues must be the roots of $1 + L(s) = 0$, with $L(s)$ read off the series diagram by `transfer_function`.

In [ ]:
T = C @ plant  # r -> [error] -> [pid] -> u -> [pendulum] -> y, theta back to the error
T.plot_diagram()

loop_tf = L.transfer_function()  # L(s) = C(s) H(s) of the wired diagram
print("closed-loop poles, from the diagram: ", np.round(np.sort_complex(np.linalg.eigvals(T.jacobian("f", "x"))), 3))
print("closed-loop poles, roots of 1 + L(s):", np.round(np.sort_complex(np.roots(np.polyadd(loop_tf.denominator, loop_tf.numerator))), 3))

The four frequency responses, back to back, in the order of the lecture. The plant $H(s)$ is torque to angle; the compensator $C(s)$ is error to command; the loop gain $L(s) = C(s)\,H(s)$ is the open chain, where the margins live; the closed loop $CL(s)$ is reference to angle, $CL = L/(1+L)$ on this SISO loop. Same `plot_bode` on each. Margins are marked only on $L$.


In [ ]:
plant.plot_bode(x_bar, margins=False, backend=BACKEND, title="H(s)")  # plant: tau -> theta
C.plot_bode(margins=False, backend=BACKEND, title="C(s)")  # compensator: e -> u
L.plot_bode(backend=BACKEND, title="L(s)")  # L = C H, open chain; margins on this plot
T.plot_bode(margins=False, backend=BACKEND, title="CL(s)")  # closed loop: r -> theta


The step response of the closed loop, from the reference $r$ to the angle $\theta$, is the linear prediction of what the loop does. Integral action makes $C(0)$ infinite, so $\theta_\infty / r \to 1$: the PID drives the tracking error to zero. A PD ($K_i = 0$) would leave a static offset. `step_info` reads rise time, settling time and overshoot off the samples; the plot prints them.

In [ ]:
time, theta = T.step_response()  # defaults: from r to y[0]
info = step_info(time, theta)
print(f"PID:  steady state = {info.steady_state:.3f}   overshoot = {info.overshoot:.1f} %")
print(f"      rise time = {info.rise_time:.2f} s   settling time = {info.settling_time:.2f} s")
T.plot_step_response(backend=BACKEND)

T_pd = PID(Kp=20.0, Ki=0.0, Kd=2.0, tau=0.05) @ plant  # no integral: leftover offset
info_pd = step_info(*T_pd.step_response())
print(f"PD:   steady state = {info_pd.steady_state:.3f}   overshoot = {info_pd.overshoot:.1f} %")
T_pd.plot_step_response(backend=BACKEND)

Finally the check that no linear tool can make: simulate the nonlinear loop. A reference of $0.3$ rad stays close to the linear regime, so the nonlinear angle settles near the linear prediction; a reference of $2$ rad does not, and the difference is what the linearization leaves out.

In [ ]:
for r in (0.3, 2.0):
    T.inputs["r"].set_nominal_value(np.array([r]))
    plant.x0 = np.zeros(2)
    traj = T.compute_trajectory(tf=15.0, verbose=False)
    print(f"r = {r}:  nonlinear final angle = {traj.x[-2, -1]:.3f}   linear prediction = {info.steady_state * r:.3f}")

T.plot_trajectory(backend=BACKEND)

## 6. Recap

| step | minilink | the math underneath |
| --- | --- | --- |
| linearize at $(\bar x, \bar u)$ | `plant.linearize(x_bar)` | $A, B, C, D$ by autodiff or finite differences |
| one channel | `of=("y", i)`, `wrt=("u", j)` | rows and columns of $C, D$ and $B$ |
| poles, zeros, gain | `plant.pzmap(x_bar)` | $\operatorname{eig}(A)$, the Rosenbrock pencil, the first Markov parameter |
| frequency response | `plant.bode`, `C.plot_bode`, `L.plot_bode`, `T.plot_bode` | $H$, then $C$, then $L=CH$, then $CL$ |
| loop gain | `L = C >> plant` | the series diagram, linearized like any system |
| margins | `L.margins()` | crossings of $0$ dB and $-180°$ |
| root locus | `L.root_locus()` | $\operatorname{eig}(A - b K c)$ over a gain sweep |
| closed loop | `T = C @ plant`, or `L @ 1`, or `feedback(L, of=..., sign=...)` | an `Error` block, a `Demux` when needed, all visible in the diagram |
| step response | `T.step_response()` | $x_{k+1} = e^{A\Delta t} x_k + \ldots$, one matrix exponential |

Every `plot_` twin takes `backend="matplotlib"` or `backend="plotly"` — this notebook passes `BACKEND` from the import cell. Every verb is also a function in `minilink.analysis` for scripts: `bode(plant, x_bar)`, `root_locus(L)`, `margins(L)`. Compensators come in two layouts from one law: `PID(...)` takes the error, `PID(..., ports="reference")` takes `r` and `y`.